In [ ]:
!pip install tinymlgen
import numpy as np
from sklearn.datasets import load_digits
import tensorflow as tf
from tensorflow.keras import layers
import tinymlgen
import tensorflow_datasets as tfds
import os
from PIL import Image

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for tinymlgen: filename=tinymlgen-0.2-py3-none-any.whl size=2226 sha256=a399bc2af965d3fa91346eb41191a18be390d980c8a68c48820d11ae52c2cecd
  Stored in directory: /root/.cache/pip/wheels/0c/46/2e/f10b7800fba560246b2c0f4317010a8c40ae3999f457a59efc
  Created wheel for hexdump: filename=hexdump-3.3-py3-none-any.whl size=8894 sha256=278c25f53d0e84336e5e01c4fe4b0d68da8182a2a5d1477021f2f5220ea6721a
  Stored in directory: /root/.cache/pip/wheels/5e/b9/b5/227a20e7e8bbdb2b17e46a087c0f0119059ee65fe8374cac18
Successfully built tinymlgen hexdump


In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')

print("GPU:", tf.config.list_physical_devices('GPU'))
print("Num GPUs:", len(physical_devices))

GPU: []
Num GPUs: 0


In [ ]:
def get_data():
    # Load MNIST dataset from tensorflow_datasets
    (train_ds, test_ds), ds_info = tfds.load(
        'mnist', split=['train', 'test'], as_supervised=True, with_info=True)

    def preprocess_data(image, label):
        image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0, 1]
        image = tf.reshape(image, (28, 28, 1))      # Reshape to (28, 28, 1)
        return image, label

    # Apply preprocessing
    train_ds_processed = train_ds.map(preprocess_data)
    test_ds_processed = test_ds.map(preprocess_data)

    # Convert datasets to NumPy arrays
    x_train_full = np.array([img.numpy() for img, label in train_ds_processed])
    y_train_full = np.array([label.numpy() for img, label in train_ds_processed])

    x_test = np.array([img.numpy() for img, label in test_ds_processed])
    y_test = np.array([label.numpy() for img, label in test_ds_processed])

    # Split x_train_full and y_train_full into training and validation sets
    TRAIN_SIZE = int(0.8 * len(x_train_full)) # 80% for training

    x_train = x_train_full[:TRAIN_SIZE]
    y_train = y_train_full[:TRAIN_SIZE]

    x_validate = x_train_full[TRAIN_SIZE:]
    y_validate = y_train_full[TRAIN_SIZE:]

    return x_train, x_test, x_validate, y_train, y_test, y_validate

print("Updated get_data() function to use TensorFlow Datasets for MNIST.")

Updated get_data() function to use TensorFlow Datasets for MNIST.


In [ ]:
def test_model(model, x_test, y_test):
    x_test = (x_test / x_test.max()).reshape((len(x_test), 28, 28, 1))
    y_pred = model.predict(x_test).argmax(axis=1)

    print('ACCURACY', (y_pred == y_test).sum() / len(y_test))

In [24]:
def get_model():
    # x_train, x_test, x_validate, y_train, y_test, y_validate = get_data()
    x_train, x_test, x_validate, y_train, y_test, y_validate = load_custom_dataset("./dataset")
    model = tf.keras.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax') # 10 classes for digits
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Create checkpoint directory if it doesn't exist
    checkpoint_dir = "./checkpoints"
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_path = f"{checkpoint_dir}/model_50epochs.keras"

    # Create checkpoint callback
    cp_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    )

    # Train for 50 epochs with checkpoint
    model.fit(x_train, y_train, epochs=100, verbose=1,
              validation_data=(x_validate, y_validate),
              callbacks=[cp_callback])

    print(f"\nModel saved to {checkpoint_path}")
    return model, x_test, y_test


In [ ]:
def test_model(model, x_test, y_test):
    # x_test is already normalized and correctly shaped from get_data() and get_model()
    y_pred = model.predict(x_test).argmax(axis=1)

    print('ACCURACY', (y_pred == y_test).sum() / len(y_test))

In [25]:
model, x_test, y_test = get_model()
test_model(model, x_test, y_test)

Loading images from ./dataset/0...
Loading images from ./dataset/1...
Loading images from ./dataset/2...
Loading images from ./dataset/3...
Loading images from ./dataset/4...
Loading images from ./dataset/5...
Loading images from ./dataset/6...
Loading images from ./dataset/7...
Loading images from ./dataset/8...
Loading images from ./dataset/9...
Loaded 1753 images
Features shape: (1753, 28, 28, 1)
Labels shape: (1753,)
Training set: 1051 images
Validation set: 351 images
Test set: 351 images
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


28/33 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2916 - loss: 2.0844
Epoch 1: val_accuracy improved from -inf to 0.76068, saving model to ./checkpoints/model_50epochs.keras
33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.3258 - loss: 2.0276 - val_accuracy: 0.7607 - val_loss: 1.0730
Epoch 2/100
24/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8403 - loss: 0.8888
Epoch 2: val_accuracy improved from 0.76068 to 0.90598, saving model to ./checkpoints/model_50epochs.keras
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8505 - loss: 0.8288 - val_accuracy: 0.9060 - val_loss: 0.4102
Epoch 3/100
25/33 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9280 - loss: 0.3446
Epoch 3: val_accuracy improved from 0.90598 to 0.95726, saving model to ./checkpoints/model_50epochs.keras
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9299 - loss: 0.3311 - val_accuracy: 0.9573 - val_loss: 0.2385
Epoch 4/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9661 - loss: 0.1927
Epoch 

In [ ]:
c_code = tinymlgen.port(model, optimize=False, pretty_print=True)
# print(c_code)
with open("file.txt", "a") as f:
  f.write(c_code)



Saved artifact at '/tmp/tmp5ipkbmpv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  136266582541520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136266582544784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136266582544208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136266582544592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136266582544976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136266582542864: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
# stop point not to let the code run automatically from here
adfadsff

NameError: name 'adfadsff' is not defined

prepare own dataset

In [ ]:
!mv dataset_refactored.zip dataset.zip && unzip dataset.zip

Archive:  dataset.zip
   creating: dataset/
  inflating: dataset/.DS_Store       
   creating: dataset/9/
 extracting: dataset/9/1766855440346_9.png  
  inflating: dataset/9/1766839487845_9.png  
  inflating: dataset/9/1766861301233_9.png  
 extracting: dataset/9/1766857594948_9.png  
  inflating: dataset/9/1766861034127_9.png  
 extracting: dataset/9/1766917515790_9.png  
 extracting: dataset/9/1766917340446_9.png  
  inflating: dataset/9/1766842687663_9.png  
 extracting: dataset/9/1766918275285_9.png  
 extracting: dataset/9/1766857883566_9.png  
 extracting: dataset/9/1766855074665_9.png  
 extracting: dataset/9/1766857573104_9.png  
 extracting: dataset/9/1766917915232_9.png  
  inflating: dataset/9/1766838015000_9.png  
 extracting: dataset/9/1766855069602_9.png  
  inflating: dataset/9/1766838020589_9.png  
  inflating: dataset/9/1766842745869_9.png  
  inflating: dataset/9/1766861024920_9.png  
 extracting: dataset/9/1766856446106_9.png  
  inflating: dataset/9/1766847446830_9.

In [ ]:
# Load images to features and labels
from sklearn.model_selection import train_test_split

def load_images_to_data(image_label, image_directory, features_data, label_data):
    list_of_files = os.listdir(image_directory)
    for file in list_of_files:
        image_file_name = os.path.join(image_directory, file)
        if ".png" in image_file_name:
            img = Image.open(image_file_name).convert("L")
            img = np.resize(img, (28,28,1))
            im2arr = np.array(img)
            im2arr = im2arr.reshape(1,28,28,1)
            features_data = np.append(features_data, im2arr, axis=0)
            label_data = np.append(label_data, [image_label], axis=0)
    return features_data, label_data

In [ ]:
folder = "./dataset/"
checkpoint_dir = "./checkpoints"

In [ ]:
# Load the saved model checkpoint
checkpoint_dir = "./checkpoints"
checkpoint_path = f"{checkpoint_dir}/model_50epochs.keras"

if os.path.exists(checkpoint_path):
    print(f"Loading model from {checkpoint_path}")
    model = tf.keras.models.load_model(checkpoint_path)
    print("Model loaded successfully!")
else:
    print(f"No checkpoint found at {checkpoint_path}. Train the model first!")


No checkpoint found at ./checkpoints/model_50epochs.keras. Train the model first!


In [ ]:
# Load custom dataset from dataset/ folder
def load_custom_dataset(dataset_folder):
    features_data = np.empty((0, 28, 28, 1))
    label_data = np.empty((0,))

    # Loop through each digit folder (0-9)
    for digit in range(10):
        digit_folder = os.path.join(dataset_folder, str(digit))
        if os.path.exists(digit_folder):
            print(f"Loading images from {digit_folder}...")
            features_data, label_data = load_images_to_data(
                digit, digit_folder, features_data, label_data
            )

    # Normalize the data
    features_data = features_data / 255.0

    print(f"Loaded {len(features_data)} images")
    print(f"Features shape: {features_data.shape}")
    print(f"Labels shape: {label_data.shape}")

    # Split into train_val_pool and test sets (e.g., 80% train_val, 20% test)
    # Use stratify to ensure class distribution is maintained across splits
    x_train_val_pool, x_test, y_train_val_pool, y_test = train_test_split(
        features_data, label_data, test_size=0.2, random_state=42, stratify=label_data
    )

    # Split train_val_pool into training and validation sets (e.g., 80% train, 20% val from the pool)
    # This means 0.25 of the remaining 80% (which is 20% of the total for validation)
    x_train, x_validate, y_train, y_validate = train_test_split(
        x_train_val_pool, y_train_val_pool, test_size=0.25, random_state=42, stratify=y_train_val_pool
    )

    print(f"Training set: {len(x_train)} images")
    print(f"Validation set: {len(x_validate)} images")
    print(f"Test set: {len(x_test)} images")

    return x_train, x_test, x_validate, y_train, y_test, y_validate

# Load the custom dataset and get the splits
custom_x_train, custom_x_test, custom_x_val, custom_y_train, custom_y_test, custom_y_val = load_custom_dataset("./dataset/")

Loading images from ./dataset/0...
Loading images from ./dataset/1...
Loading images from ./dataset/2...
Loading images from ./dataset/3...
Loading images from ./dataset/4...
Loading images from ./dataset/5...
Loading images from ./dataset/6...
Loading images from ./dataset/7...
Loading images from ./dataset/8...
Loading images from ./dataset/9...
Loaded 1753 images
Features shape: (1753, 28, 28, 1)
Labels shape: (1753,)
Training set: 1051 images
Validation set: 351 images
Test set: 351 images


In [ ]:
# Continue training on custom dataset
# The data is already split into train, validation, and test sets by load_custom_dataset.
# The print statements for training and validation set sizes are now within load_custom_dataset.

# Load the saved model checkpoint if not already loaded
# This addresses the NameError if 'model' was not previously defined or loaded.
checkpoint_path_initial = f"{checkpoint_dir}/model_50epochs.keras"
if 'model' not in locals() and 'model' not in globals():
    if os.path.exists(checkpoint_path_initial):
        print(f"Loading initial model from {checkpoint_path_initial}")
        model = tf.keras.models.load_model(checkpoint_path_initial)
        print("Initial model loaded successfully!")
    else:
        # Fallback if the initial model isn't found, though it should be.
        print("Initial model checkpoint not found. Re-running get_model() to train and load a new one.")
        model, _, _ = get_model() # This will retrain and save a new model


# Create checkpoint callback for continued training
checkpoint_path_custom = f"{checkpoint_dir}/model_custom_trained.keras"
cp_callback_custom = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path_custom,
    save_best_only=True,
    monitor='val_accuracy',
    verbose=1
)

# Continue training on custom dataset
print("\nContinuing training on custom dataset...")
history = model.fit(
    custom_x_train, custom_y_train,
    epochs=20,
    verbose=1,
    validation_data=(custom_x_val, custom_y_val),
    callbacks=[cp_callback_custom]
)

print(f"\nModel trained on custom dataset and saved to {checkpoint_path_custom}")

# Evaluate the model on the custom test set
print("\nEvaluating model on custom test set...")
model_loss, model_accuracy = model.evaluate(custom_x_test, custom_y_test, verbose=0)
print(f"Custom Test Accuracy: {model_accuracy:.4f}")


Continuing training on custom dataset...


NameError: name 'model' is not defined